In [1]:
# 导入模块
import sys
import json
import pandas as pd
import numpy as np
import tensorflow as tf
sys.path.append("..")
from utils.set_fonts import apply_times_with_cn

en_fp, cn_fp = apply_times_with_cn(cn_font_path=r"C:\Windows\Fonts\msyh.ttc")

Applied font.family: ['Times New Roman', 'Microsoft YaHei', 'DejaVu Sans']
findfont Times -> C:\Windows\Fonts\times.ttf
findfont CN -> C:\Windows\Fonts\msyh.ttc


In [19]:
import torch
print(torch.__version__)

2.2.2+cpu


In [2]:
import numpy as np
import pandas as pd
from data.data_processor import TabularDataProcessor

# ============================================
# 构造一个测试DataFrame
# ============================================

df = pd.DataFrame({
    "Age": [18, 25, 31, 40, 55, 35, 19, 24, 30, 22, 38, 40, 45, 60, 38],
    "BMI": [20.1, 19.8, 26.3, 30.2, 18.5, 18.4, 20.1, 22.5, 26.1, 30.1, 20, 19, 22, 24, 18],
    "Glucose": [90, 120, 110, 85, 95, 80, 100, 105, 110, 115, 120, 111, 145, 127, 134],
    "Gender": ["M", "F", "F", "M", "M", "M", "M", "F","F", "M", "M", "F", "F", "F", "M"],
    "Smoking": ["Yes", "No", "No", "Yes", "No", "No", "No", "Yes", "Yes", "No", "Yes", "No", "Yes", "Yes", "No"],
    "label": [0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0]
})

num_cols = ["Age", "BMI", "Glucose"]
cat_cols = ["Gender", "Smoking"]

X = df.drop(columns="label")
y = df["label"]

df.head(15)

,Age,BMI,Glucose,Gender,Smoking,label
0,18,20.1,90,M,Yes,0
1,25,19.8,120,F,No,0
2,31,26.3,110,F,No,0
3,40,30.2,85,M,Yes,0
4,55,18.5,95,M,No,1
5,35,18.4,80,M,No,1
6,19,20.1,100,M,No,0
7,24,22.5,105,F,Yes,0
8,30,26.1,110,F,Yes,1
9,22,30.1,115,M,No,1


In [16]:
import numpy as np
import pandas as pd

def detect_precision(series):
    """
    检测一个数值列中所有非缺失值的小数位数，返回最大小数位数。
    """
    # 去除缺失值
    clean_vals = series.dropna()
    if len(clean_vals) == 0:
        return 0  # 全缺失则默认0
    # 使用字符串转换获取小数位数，避免科学计数法
    max_decimals = 0
    for v in clean_vals:
        # 如果值是整数或浮点，转为字符串，去掉末尾的0，再分割
        s = f"{v:.12f}".rstrip('0').rstrip('.')  # 先固定12位小数，避免科学计数法
        if '.' in s:
            decimals = len(s.split('.')[1])
        else:
            decimals = 0
        if decimals > max_decimals:
            max_decimals = decimals
    return max_decimals

def get_precision_dict(X_train, categorical_features):
    """
    为训练数据中的每个数值列计算精度。
    X_train: DataFrame
    categorical_features: 分类特征列名列表（或索引）
    返回: 字典 {列名: 精度小数位数}
    """
    precision_dict = {}
    for col in X_train.columns:
        if col not in categorical_features:
            precision_dict[col] = detect_precision(X_train[col])
    return precision_dict

def restore_precision(X_generated, precision_dict):
    """
    将生成的DataFrame的数值列四舍五入到原始精度。
    返回新的DataFrame。
    """
    X_restored = X_generated.copy()
    for col, decimals in precision_dict.items():
        if col in X_restored.columns:
            X_restored[col] = X_restored[col].round(decimals)
    return X_restored

In [18]:
from imblearn.over_sampling import SMOTENC
from collections import Counter

counts = Counter(y)
count_0 = counts[0]
count_1 = counts[1]

print(f"count_0: {count_0}, count_1: {count_1}")
target_counts = {0: count_0, 1: (count_1 + 2)}    
print(target_counts)

precision_dict = get_precision_dict(X, cat_cols)

model = SMOTENC(
    categorical_features=cat_cols, 
    sampling_strategy=target_counts,
    random_state=42,
    )

X_resampled, y_resampled = model.fit_resample(X, y)

X_resampled_restored = restore_precision(pd.DataFrame(X_resampled, columns=X.columns), precision_dict)

X_syn = X_resampled_restored.iloc[len(X):]

print(X_syn)

count_0: 8, count_1: 7
{0: 8, 1: 9}
    Age   BMI  Glucose Gender Smoking
15   50  18.6       99      M     Yes
16   44  21.6      101      M      No


In [ ]:
from imblearn.over_sampling import SMOTENC
from collections import Counter


def generate(fold_data, n_samples):

    X = fold_data.X_train
    y = fold_data.y_train
    counts = Counter(y)
    count_0 = counts[0]
    count_1 = counts[1]
    target_counts = {0: count_0, 1: (count_1 + n_samples)}    

    model = SMOTENC(
        categorical_features=fold_data.cat_cols, 
        sampling_strategy=target_counts,
        random_state=fold_data.seed
        )
    
    X_resampled, y_resampled = model.fit_resample(X, y, shuffle=False)

    X_syn = X_resampled[len(X):]

    return X_syn
    

In [2]:
import numpy as np
import pandas as pd
from data.data_processor import TabularDataProcessor

# ============================================
# 构造一个测试DataFrame
# ============================================

df = pd.DataFrame({
    "Age": [18, 25, 31, 40, 55],
    "BMI": [20.1, 19.8, 26.3, 30.2, 18.5],
    "Glucose": [90, 120, 110, 85, 95],
    "Gender": ["M", "F", "F", "M", "M"],
    "Smoking": ["Yes", "No", "No", "Yes", "No"],
})

num_cols = ["Age", "BMI", "Glucose"]
cat_cols = ["Gender", "Smoking"]
df.head()

,Age,BMI,Glucose,Gender,Smoking
0,18,20.1,90,M,Yes
1,25,19.8,120,F,No
2,31,26.3,110,F,No
3,40,30.2,85,M,Yes
4,55,18.5,95,M,No


In [5]:
from synthcity.plugins import Plugins

model = Plugins().get("tvae")
model.fit(df)
X_tvae = model.generate(count=5)


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


[2026-07-04T20:59:08.327464+0800][20304][CRITICAL] Error importing TabularGoggle: No module named 'torch.utils._import_utils'
[2026-07-04T20:59:08.336924+0800][20304][CRITICAL] module disabled: d:\software\anaconda3\envs\torch\Lib\site-packages\synthcity\plugins\generic\plugin_goggle.py
 25%|██▌       | 250/1000 [00:19<00:57, 13.03it/s]


In [7]:
print(type(X_tvae))
X_syn = X_tvae.dataframe()
X_syn.head()

<class 'synthcity.plugins.core.dataloader.GenericDataLoader'>


,Age,BMI,Glucose,Gender,Smoking
0,55,18.5,85,M,Yes
1,18,20.1,90,M,Yes
2,55,18.5,95,M,No
3,40,30.2,85,M,Yes
4,55,18.5,95,M,No


In [13]:
from ctgan import CTGAN

discrete_columns = ['Gender', 'Smoking']

model = CTGAN(epochs=300)
model.fit(df, discrete_columns=discrete_columns)
X_syn = model.sample(5)

In [5]:
from synthcity.plugins import Plugins

tabddpm = Plugins().get("ddpm")
tabddpm.fit(df)

syn = tabddpm.generate(5).dataframe()

syn.head()

[2026-07-04T21:36:39.483844+0800][1064][CRITICAL] Error importing TabularGoggle: No module named 'torch.utils._import_utils'
[2026-07-04T21:36:39.487375+0800][1064][CRITICAL] module disabled: d:\software\anaconda3\envs\torch\Lib\site-packages\synthcity\plugins\generic\plugin_goggle.py
Epoch: 100%|██████████| 1000/1000 [00:22<00:00, 45.04it/s, loss=0.561]


,Age,BMI,Glucose,Gender,Smoking
0,31,26.3,110,F,No
1,40,30.2,85,M,Yes
2,25,19.8,120,F,No
3,31,18.5,90,M,Yes
4,18,20.1,90,M,Yes
